In [ ]:
import re
import pandas as pd

# 1. Leer el archivo original de 2025-2024 como texto plano
with open('llegadas_2025_2024_clean.csv', 'r', encoding='utf-8-sig', errors='ignore') as f:
    lineas = f.readlines()

lineas_datos = [l.strip() for l in lineas if l.strip() != ""]
if "RO" in lineas_datos[0]:
    lineas_datos = lineas_datos[1:]

# 2. ORDEN INVERTIDO DE LAS COLUMNAS (De más nuevo a más viejo: Dic 2025 -> Ene 2024)
meses_columnas = [
    "2025-12", "2025-11", "2025-10", "2025-09", "2025-08", "2025-07",
    "2025-06", "2025-05", "2025-04", "2025-03", "2025-02", "2025-01",
    "2024-12", "2024-11", "2024-10", "2024-09", "2024-08", "2024-07",
    "2024-06", "2024-05", "2024-04", "2024-03", "2024-02", "2024-01"
]

matriz_paises = {}

# 3. Extraer la serie temporal de cada país
for linea in lineas_datos[1:]:
    if "Pasajeros Totales" in linea:
        continue

    paises_encontrados = re.findall(r'[A-ZÁÉÍÓÚÑüÜ]{3,}(?:\s+[A-ZÁÉÍÓÚÑüÜ]+)*', linea)
    if not paises_encontrados:
        continue
    pais = paises_encontrados[0].strip()

    numeros = re.findall(r'"([\d\.]+)"', linea)
    if not numeros:
        numeros = re.findall(r'\b\d+(?:\.\d+)*\b', linea)

    valores_meses = []
    for num in numeros:
        # Mantenemos el parche de seguridad para los ceros borrados por Excel
        if '.' in num and len(num.split('.')[-1]) == 2:
            num = num + "0"

        num_limpio = num.replace('.', '')
        if num_limpio.isdigit():
            valores_meses.append(int(num_limpio))

    matriz_paises[pais] = valores_meses

# 4. Construir la estructura temporal mes a mes
paises_top = {'ESPAÑA', 'SPAIN', 'FRANCIA', 'FRANCE', 'ITALIA', 'ITALY',
              'REINO UNIDO', 'UNITED KINGDOM', 'UK', 'PAISES BAJOS', 'HOLANDA',
              'NETHERLANDS', 'ALEMANIA', 'GERMANY'}

datos_historicos_finales = []

for idx_mes, mes_nombre in enumerate(meses_columnas):

    def extraer_valor_mes(nombre_pais):
        for k, lista_valores in matriz_paises.items():
            if nombre_pais in k and idx_mes < len(lista_valores):
                return lista_valores[idx_mes]
        return 0

    nac = extraer_valor_mes('ESPAÑA') or extraer_valor_mes('SPAIN')
    fr = extraer_valor_mes('FRANCIA') or extraer_valor_mes('FRANCE')
    it = extraer_valor_mes('ITALIA') or extraer_valor_mes('ITALY')
    uk = extraer_valor_mes('REINO UNIDO') or extraer_valor_mes('UNITED KINGDOM') or extraer_valor_mes('UK')
    nl = extraer_valor_mes('PAISES BAJOS') or extraer_valor_mes('HOLANDA') or extraer_valor_mes('NETHERLANDS')
    de = extraer_valor_mes('ALEMANIA') or extraer_valor_mes('GERMANY')

    resto_del_mundo = 0
    for k, lista_valores in matriz_paises.items():
        if not any(p in k for p in paises_top) and idx_mes < len(lista_valores):
            resto_del_mundo += lista_valores[idx_mes]

    internacionales = 0
    for k, lista_valores in matriz_paises.items():
        if 'ESPAÑA' not in k and 'SPAIN' not in k and idx_mes < len(lista_valores):
            internacionales += lista_valores[idx_mes]

    total_aeropuerto = nac + internacionales

    datos_historicos_finales.append({
        'Mes': mes_nombre,
        'Volumen total llegadas aeropuerto': total_aeropuerto,
        'Volumen llegadas Nacionales': nac,
        'Volumen llegadas Internacionales': internacionales,
        'Volumen llegadas Francia': fr,
        'Volumen llegadas Italia': it,
        'Volumen llegadas Reino Unido': uk,
        'Volumen llegadas Países Bajos': nl,
        'Volumen llegadas Alemania': de,
        'Volumen llegadas Resto del Mundo': resto_del_mundo
    })

# 5. Convertir a DataFrame, ordenar cronológicamente (Ene 2024 -> Dic 2025) y guardar
df_temporal = pd.DataFrame(datos_historicos_finales)
df_temporal = df_temporal.sort_values(by='Mes').reset_index(drop=True)

# Guardar con el nombre correspondiente
df_temporal.to_csv('2025-2024.csv', index=False, encoding='utf-8-sig')


¡Terminado! Tu último bloque ha sido guardado como: '2025-2024.csv'


,Mes,Volumen total llegadas aeropuerto,Volumen llegadas Nacionales,Volumen llegadas Internacionales,Volumen llegadas Francia,Volumen llegadas Italia,Volumen llegadas Reino Unido,Volumen llegadas Países Bajos,Volumen llegadas Alemania,Volumen llegadas Resto del Mundo
0,2024-01,296080,142914,153166,28030,37752,23106,8244,10034,46000
1,2024-02,331509,153474,178035,35136,34335,34119,13978,11557,48910
2,2024-03,393499,184636,208863,42526,40276,38083,16109,14850,57019
3,2024-04,426954,212514,214440,51364,40258,34862,11613,15344,60999
4,2024-05,422435,212716,209719,47699,38493,35307,11995,15482,60743
5,2024-06,382507,193805,188702,41673,35998,29893,11101,13585,56452
6,2024-07,382492,199606,182886,38748,38708,27472,7654,14396,55908
7,2024-08,379221,198769,180452,33094,40167,25460,8086,14918,58727
8,2024-09,404137,200649,203488,39318,37858,31904,13921,16551,63936
9,2024-10,422638,199004,223634,45557,44341,34308,15641,17021,66766
